In [9]:
import torch
import os
import torch.nn as nn

# Loads the preprocessed dataset
data = torch.load("/home/jovyan/megan/labb3/coco_train_preprocessed.pt")

# Extracts the saved data
samples = data["samples"]
vocab = data["vocab"]
idx_to_word = data["idx_to_word"]
max_length = data["max_length"]

class ImageCaptionModel(nn.Module):
    def __init__(self, vocab_size, max_length):
        super(ImageCaptionModel, self).__init__()
        
        # Images network
        self.image_embedding = nn.Sequential(
            nn.Linear(25088, 256), # Reduces image features into feature size 256
            nn.ReLU() # ReLU to add non-linearity
        )

        # --SEQUENCE MODEL FOR LANGUAGE PROCESSING--
        # Captions network
        self.caption_embedding = nn.Embedding(
            num_embeddings=vocab_size, # Num of words in vocabulary
            embedding_dim=256 # Size of the word vectors
        )

        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=256,
            batch_first=True
        )

        # Output layer (prediction)
        self.output_layer = nn.Linear(256, vocab_size)

    def forward(self, image_features, captions):
        image_embedding = self.image_embedding(image_features) # Converts CNN features into image embeds
        
        caption_embedding = self.caption_embedding(captions) # Converts token ids into word embeds
        caption_embedding, _ = self.lstm(caption_embedding) # Passes captions through lstm

        # --COMBINING IMAGE AND TEXT DATA--
        # Combining image and text
        image_embedding = image_embedding.unsqueeze(1)
        image_embedding = image_embedding.repeat(1, caption_embedding.size(1), 1)

        combined_embeddings = image_embedding + caption_embedding

        # Output word prediction
        output = self.output_layer(combined_embeddings)
        return output

# Model
model = ImageCaptionModel(
    vocab_size=len(vocab),
    max_length=max_length
)

# Loss function
criterion = nn.CrossEntropyLoss(
    ignore_index=vocab["<pad>"]
)

# Optimizer (picked Adam as its fast and stable)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Test
print(model)

ImageCaptionModel(
  (image_embedding): Sequential(
    (0): Linear(in_features=25088, out_features=256, bias=True)
    (1): ReLU()
  )
  (caption_embedding): Embedding(10307, 256)
  (lstm): LSTM(256, 256, batch_first=True)
  (output_layer): Linear(in_features=256, out_features=10307, bias=True)
)
